# VP Noise Scheduler Diagnostics

Visual check for large-$T$ behavior of the closed-form forward process:

$$X_t = \alpha_t X_0 + \sigma_t Z,\quad \alpha_t = e^{-\beta t/2},\ \sigma_t=\sqrt{1-e^{-\beta t}},\ Z\sim\mathcal N(0,1).$$

As $t\to\infty$, this converges to $\mathcal N(0,1)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from implied_volatility_diffusion import VPNoiseScheduler

rng = np.random.default_rng(42)
scheduler = VPNoiseScheduler(beta=1.0)
x0 = np.full(200_000, 3.0)
times = [0.0, 0.5, 2.0, 5.0, 20.0]

means, variances = [], []
samples = {}
for t in times:
    xt = scheduler.add_noise(x0, t=t, rng=rng)
    samples[t] = xt
    means.append(np.mean(xt))
    variances.append(np.var(xt))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xgrid = np.linspace(-4, 4, 300)
std_pdf = (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * xgrid**2)

for t in times:
    axes[0].hist(samples[t], bins=80, density=True, alpha=0.35, label=f'T={t}')
axes[0].plot(xgrid, std_pdf, 'k--', linewidth=2, label='N(0,1) pdf')
axes[0].set_title('Distribution as T increases')
axes[0].set_xlim(-4, 4)
axes[0].legend()

axes[1].plot(times, means, marker='o', label='mean')
axes[1].plot(times, variances, marker='o', label='variance')
axes[1].axhline(0.0, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Moment convergence')
axes[1].set_xlabel('T')
axes[1].legend()

plt.tight_layout()
plt.show()
